# Colab: data → Qwen3-VL-32B LoRA (LLaMA-Factory)

Run top to bottom. **`huggingface-cli login`** if the Hub model is gated. Data: **`/content/out`**.

**Checkpoint (LoRA + trainer files):** `/content/LLaMA-Factory/saves/qwen3vl32-rtfm-lora` — Colab **deletes `/content` on disconnect**; run the last cell after training to copy to Drive, or zip/download manually.

In [1]:
import importlib.util
import subprocess
from pathlib import Path

assert Path("/content").is_dir()
assert importlib.util.find_spec("google.colab") is not None
_r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
print(_r.stdout.strip() or _r.stderr)

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-89850228-b50a-d801-5981-91eaae3400dc)


In [2]:
import shutil
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
src, dest = Path("/content/drive/MyDrive/out"), Path("/content/out")
assert (src / "manifest.jsonl").is_file(), src

def _ign(_d, names):
    s = {"Icon\r", ".DS_Store", "desktop.ini"}
    return [n for n in names if n in s or n.startswith("._")]

shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(src, dest, dirs_exist_ok=True, ignore=_ign)
print(dest)

Mounted at /content/drive
/content/out


In [3]:
import json
import random
import shutil
from pathlib import Path

DATA_ROOT = Path("/content/out")
MANIFEST = DATA_ROOT / "manifest.jsonl"
WORK = Path("/content/lf_data")
VAL_FRAC, SEED = 0.15, 42
NAME = "rtfm_qwen_sft"
# Cap frames per example: pooling can produce 100+ images (e.g. band × many snippets),
# which blows vision tokens, VRAM, and cutoff_len. Subsample in time, keep scores aligned.
MAX_IMAGES = 32
random.seed(SEED)
WORK.mkdir(parents=True, exist_ok=True)
train_j, eval_j = WORK / f"{NAME}_train.json", WORK / f"{NAME}_eval.json"

rows = []
with open(MANIFEST, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
miss = []
for r in rows:
    for rel in r["images"]:
        if not (DATA_ROOT / rel).is_file():
            miss.append(rel)
assert not miss, miss[:5]

def user_prompt_from_scores(scores: list[float]) -> str:
    s = ", ".join(f"{x:.4f}" for x in scores)
    return (
        f"Anomaly scores for each shown frame (temporal order, RTFM logits): [{s}]\n"
        "Describe the anomalous activity."
    )

def subsample_frames(rel_imgs: list[str], scores: list[float], max_n: int):
    assert len(rel_imgs) == len(scores)
    n = len(rel_imgs)
    if n <= max_n:
        return rel_imgs, scores
    if max_n == 1:
        idx = [0]
    else:
        idx = sorted({int(round(i * (n - 1) / (max_n - 1))) for i in range(max_n)})
    return [rel_imgs[i] for i in idx], [scores[i] for i in idx]

def to_lf(r):
    rels = list(r["images"])
    sc = list(r.get("scores", []))
    if len(sc) != len(rels):
        raise ValueError(f"scores/images length mismatch: {len(sc)} vs {len(rels)} id={r.get('id')}")
    rels, sc = subsample_frames(rels, sc, MAX_IMAGES)
    imgs = [str((DATA_ROOT / rel).resolve()) for rel in rels]
    ph = "\n".join("<image>" for _ in imgs)
    u = f"{ph}\n{user_prompt_from_scores(sc)}"
    a = None
    for t in r["conversations"]:
        if t["from"] == "assistant":
            a = t["value"]
            break
    return {
        "messages": [
            {"role": "system", "content": r["system"]},
            {"role": "user", "content": u},
            {"role": "assistant", "content": a},
        ],
        "images": imgs,
    }

by = {}
for r in rows:
    by.setdefault(r["video_id"], []).append(r)
vids = sorted(by)
random.shuffle(vids)
n_val = max(1, int(round(len(vids) * VAL_FRAC)))
val = set(vids[:n_val])
tr_r, ev_r = [], []
for vid, rs in by.items():
    (ev_r if vid in val else tr_r).extend(rs)
with open(train_j, "w", encoding="utf-8") as f:
    json.dump([to_lf(r) for r in tr_r], f, ensure_ascii=False)
with open(eval_j, "w", encoding="utf-8") as f:
    json.dump([to_lf(r) for r in ev_r], f, ensure_ascii=False)
for path, rs in ((DATA_ROOT / "manifest_train.jsonl", tr_r), (DATA_ROOT / "manifest_eval.jsonl", ev_r)):
    with open(path, "w", encoding="utf-8") as f:
        for r in rs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
(DATA_ROOT / "manifest_split.json").write_text(
    json.dumps(
        {"val_video_ids": sorted(val), "n_train": len(tr_r), "n_eval": len(ev_r)}, indent=2
    ),
    encoding="utf-8",
)
print(len(tr_r), "train", len(ev_r), "eval", train_j)

270 train 45 eval /content/lf_data/rtfm_qwen_sft_train.json


In [4]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

LF = Path("/content/LLaMA-Factory")
NAME = "rtfm_qwen_sft"
WORK = Path("/content/lf_data")
train_j = WORK / f"{NAME}_train.json"
eval_j = WORK / f"{NAME}_eval.json"

if not LF.is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/hiyouga/LLaMA-Factory.git", str(LF)],
        check=True,
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{LF}[metrics]"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "bitsandbytes"],
    check=True,
)
dd = LF / "data"
dd.mkdir(parents=True, exist_ok=True)
shutil.copy(train_j, dd / train_j.name)
shutil.copy(eval_j, dd / eval_j.name)
info_p = dd / "dataset_info.json"
info = json.loads(info_p.read_text(encoding="utf-8")) if info_p.is_file() else {}
tags = {
    "role_tag": "role",
    "content_tag": "content",
    "user_tag": "user",
    "assistant_tag": "assistant",
    "system_tag": "system",
}
for suf in ("_train", "_eval"):
    info[NAME + suf] = {
        "file_name": (train_j if suf == "_train" else eval_j).name,
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": tags,
    }
info_p.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
print(LF)

/content/LLaMA-Factory


In [5]:
from pathlib import Path

LF = Path("/content/LLaMA-Factory")
NAME = "rtfm_qwen_sft"
yp = LF / "train.yaml"
yp.write_text(
    f"""
model_name_or_path: Qwen/Qwen3-VL-32B-Instruct
trust_remote_code: true
# Slightly lower res => fewer image tokens; helps stay under cutoff_len and saves VRAM.
image_max_pixels: 147456
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_target: all
dataset: {NAME}_train
eval_dataset: {NAME}_eval
template: qwen3_vl_nothink
# Must exceed longest tokenized row (e.g. ~8274 with many frames). 16384 OOMs on CE logits (~5 GiB).
# 10240 keeps headroom vs 8192 but cuts cross_entropy/logit memory vs 16384.
cutoff_len: 10240
preprocessing_num_workers: 8
dataloader_num_workers: 2
output_dir: /content/LLaMA-Factory/saves/qwen3vl32-rtfm-lora
logging_steps: 5
save_steps: 50
eval_strategy: steps
eval_steps: 50
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none
per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 2.0e-5
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
bf16: true
ddp_timeout: 180000000
""".strip()
    + "\n",
    encoding="utf-8",
)
print(yp)

/content/LLaMA-Factory/train.yaml


In [6]:
# !huggingface-cli login
import os
# Reduces allocator fragmentation (PyTorch suggestion on OOM).
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
%cd /content/LLaMA-Factory
!llamafactory-cli train train.yaml

/content/LLaMA-Factory
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-04-06 00:05:00] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 1.47kB [00:00, 4.45MB/s]
[INFO|configuration_utils.py:667] 2026-04-06 00:05:00,366 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-32B-Instruct/s

In [7]:
import shutil
from pathlib import Path

src = Path("/content/LLaMA-Factory/saves/qwen3vl32-rtfm-lora")
dst = Path("/content/drive/MyDrive/qwen3vl32_rtfm_lora")
assert src.is_dir(), f"train first — missing {src}"
assert Path("/content/drive/MyDrive").is_dir(), "mount Drive first"
shutil.copytree(src, dst, dirs_exist_ok=True)
print("copied to", dst)

copied to /content/drive/MyDrive/qwen3vl32_rtfm_lora
